# Productionize PDF Search
This sample demonstrates how to set up a robust and automated system for searching and analyzing PDF documents using LangDB. 

## Tables
* `files` will store the pdf chunks
* `files_embeddings` will store the generated embeddings

In [ ]:
CREATE TABLE files (
    id UUID DEFAULT generateUUIDv4(),
    name String,
    e_tag String,
    content String,
) engine = MergeTree
order by name;

In [ ]:
CREATE TABLE files_embeddings (
    id UUID,
    name String,
    embeddings `Array`(`Float32`),
) 
engine = MergeTree
order by name;

## Background Tasks
Three main tasks are spawned at regular intervals to ensure the database reflects the current state of the files in the specified directory:
* The `load_files` task is responsible for periodically scanning the directory for new files, loading their contents, and inserting relevant metadata into the files table.
* The `generate_embeddings` task then takes these file contents and generates embeddings, which are inserted into the files_embeddings table.
* Finally, the `delete_embeddings task` ensures data integrity by removing records from both the files and files_embeddings tables if the files are either deleted or modified in the directory.

These tasks run concurrently, with specified limits on their execution frequency and pool size, ensuring efficient and consistent data handling.
  

In [4]:
SPAWN TASK load_files
BEGIN
    INSERT INTO files(name, e_tag, content)
    SELECT JSONExtractString(metadata, 'name'), JSONExtractString(metadata, 'e_tag'), content
    FROM load_pdf_text((
        SELECT concat('file:///', name), name, e_tag FROM list_files('file:///var/lib/clickhouse/user_files')
        WHERE name NOT IN (
            SELECT name FROM files
        )
        LIMIT 5
    ))
END
EVERY 5 SECOND
WITH MAX_POOL_SIZE 5;

Spawned Task: load_files with id: 4d6aadf8-cbb3-466a-9be1-280a77248783

In [5]:
SPAWN TASK generate_embeddings
BEGIN
    INSERT INTO files_embeddings(id, name, embeddings)
    SELECT f.id, f.name, embed(content) FROM files AS f LEFT JOIN files_embeddings AS fe ON f.id = fe.id
    WHERE f.id != fe.id
    ORDER BY f.id
    LIMIT 5
END 
EVERY 5 second;

Spawned Task: generate_embeddings with id: 23d3e6dd-e9af-4a1a-85f6-e6a947ea87f8

In [6]:
SPAWN TASK delete_embeddings
BEGIN
    DELETE FROM files
    WHERE name NOT IN (
        SELECT name FROM list_files('file:///var/lib/clickhouse/user_files')
    );
    DELETE FROM files
    WHERE e_tag NOT IN (
        SELECT e_tag FROM list_files('file:///var/lib/clickhouse/user_files')
    );
    DELETE FROM files_embeddings
    WHERE name NOT IN (
        SELECT name FROM files
    );
END
EVERY 10 SECOND;

Spawned Task: delete_embeddings with id: f377e67e-ec29-4ea4-afca-dad355ea570e

## Model, Prompt, and Endpoint Creation

In [ ]:
CREATE ENDPOINT similar(query String "Query to search similar sections in pdf documents") AS
WITH tbl AS (
    SELECT CAST(embed($query) AS `Array`(`Float32`)) AS query
)
SELECT 
    f.id as id, 
    content, 
    cosineDistance(embeddings, query) AS similarity 
FROM 
    files_embeddings AS fe 
JOIN 
    files AS f ON f.id = fe.id
CROSS JOIN
    tbl 
ORDER BY 
    similarity DESC 
LIMIT 5

In [ ]:
CREATE MODEL IF NOT EXISTS files_query(
  input
) ENGINE = OpenAI(api_key = 'sk-proj-xxx', model_name = 'gpt-3.5-turbo')
PROMPT (system "Use the tool 'similar' to query for similar information based on user query to assist with quickly searching the documents.",
  human "{{input}}")
TOOLS (similar)
SETTINGS retries = 1;

In [19]:
SELECT files_query('tell me in brief about New Delhi')

New Delhi is the capital city of India. It is known for its historical landmarks, cultural attractions, and political significance. New Delhi is home to various educational institutions, sports complexes, and favorable weather conditions. The city experiences varying temperatures throughout the year with average maximum temperatures ranging from 20.1Â°C to 39.9Â°C and average minimum temperatures ranging from 3.5Â°C to 26.4Â°C. Additionally, New Delhi receives moderate rainfall with an average of 774.4 mm annually.